In [3]:
import cv2
import numpy as np
import os
from collections import Counter
import torch

class BuscadorDeImagensGPU:
    def __init__(self, db_folder, color_space='gray', aplicar_quantizacao_imagem=False, aplicar_filtro=False, bins=256):
        self.db_folder = db_folder
        self.color_space = color_space
        self.bins = bins
        self.aplicar_quantizacao_imagem = aplicar_quantizacao_imagem
        self.aplicar_filtro = aplicar_filtro

        # Define o dispositivo (GPU se disponível)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        # ATRIBUTOS QUE SERÃO PREENCHIDOS NO INIT
        self.filenames = []
        self.db_tensor = None

        # INICIALIZAÇÃO AUTOMÁTICA
        print(f"[*] Inicializando Buscador no dispositivo: {self.device}")
        self._preparar_dataset_interno()

    def _extrair_pdf_unitario(self, image_path):
        img = cv2.imread(image_path)
        if img is None: return None

        if self.aplicar_filtro:
            img = cv2.GaussianBlur(img, (5, 5), 0)

        if self.aplicar_quantizacao_imagem:
            fator = 256 // self.bins
            img = (img // fator) * fator

        if self.color_space == 'gray':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            hist = cv2.calcHist([img], [0], None, [self.bins], [0, 256])
        elif self.color_space == 'rgb':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            hist = cv2.calcHist([img], [0, 1, 2], None, [self.bins]*3, [0, 256]*3)
        elif self.color_space == 'hsv':
            img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
            hist = cv2.calcHist([img], [0, 1], None, [self.bins, self.bins], [0, 180, 0, 256])

        cv2.normalize(hist, hist, alpha=1, norm_type=cv2.NORM_L1)
        return hist.flatten().astype(np.float32)

    def _preparar_dataset_interno(self):
        self.filenames = sorted([f for f in os.listdir(self.db_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
        features = []

        print(f"[#] Extraindo características de {len(self.filenames)} imagens. Aguarde...")
        for f in self.filenames:
            pdf = self._extrair_pdf_unitario(os.path.join(self.db_folder, f))
            if pdf is not None:
                features.append(pdf)
            else:
                self.filenames.remove(f)

        self.db_tensor = torch.tensor(np.array(features), device=self.device)
        print(f"[V] Dataset preparado e carregado na GPU. Tamanho da matriz: {self.db_tensor.shape}")

    def buscar(self, query_path, N=2, metrica="euclidiana"):
        query_pdf = self._extrair_pdf_unitario(query_path)
        if query_pdf is None: return []

        query_tensor = torch.tensor(query_pdf, device=self.device)

        if metrica == 'euclidiana':
            distancias = torch.norm(self.db_tensor - query_tensor, dim=1)
        elif metrica == 'bhattacharyya':
            distancias = torch.sqrt(1 - torch.sum(torch.sqrt(self.db_tensor * query_tensor), dim=1))
        else:
            distancias = torch.norm(self.db_tensor - query_tensor, dim=1)

        valores, indices = torch.topk(distancias, k=min(N + 1, len(self.filenames)), largest=False)

        resultados = []
        query_filename = os.path.basename(query_path)

        for i in range(len(indices)):
            idx = indices[i].item()
            dist = valores[i].item()
            nome = self.filenames[idx]

            if query_filename != nome:
                resultados.append((dist, nome))

        return resultados[:N]

    def extrair_classe_do_nome(self, filename):
        return filename.split('_')[0].lower()

    def avaliar_taxa_acerto(self, N=3, metrica="euclidiana"):

        queries_filenames = [f for f in self.filenames if "_001." in f]

        acertos = 0
        total = len(queries_filenames)

        print(f"[*] Avaliando {total} protótipos via GPU...")
        for query_name in queries_filenames:
            query_path = os.path.join(self.db_folder, query_name)
            classe_real = self.extrair_classe_do_nome(query_name)

            # Busca usando o tensor da GPU
            resultados = self.buscar(query_path, N, metrica)

            classes_recuperadas = [self.extrair_classe_do_nome(res[1]) for res in resultados]

            if classes_recuperadas:
                # Votação majoritária
                classe_predita = Counter(classes_recuperadas).most_common(1)[0][0]
                if classe_predita == classe_real:
                    acertos += 1

        taxa = acertos / total if total > 0 else 0
        print(f"[V] Avaliação concluída. Taxa de acerto: {taxa:.2%}")
        return taxa

In [ ]:
import pandas as pd
import os
from datetime import datetime

# Lista para armazenar os dados das linhas da tabela
dados_tabela = []

pasta_banco = 'Vistex'
nome_arquivo_log = "resultados_vistex_log.txt"
nome_arquivo_csv = "resultados_dataframe_vistex.csv"

# Loop principal
for classe_n in range(1, 55):
    agora = datetime.now().strftime("%d/%m/%Y %H:%M:%S")

    with open(nome_arquivo_log, "a", encoding="utf-8") as arquivo_log:
        def print_log(mensagem):
            print(mensagem)
            arquivo_log.write(str(mensagem) + "\n")

        print_log(f"\n{'='*60}\nINÍCIO DOS TESTES CBIR - DATA E HORA: {agora}\n{'='*60}")


        for c_idx in range(classe_n, classe_n + 1):
            imagem_consulta = f'Vistex/c{c_idx:03d}_001.png'

            for metodo_busca in ["rgb", "gray", "hsv"]:
                for aplicar_filtro in [False, True]:
                    for aplicar_quantizacao_imagem in [False, True]:
                        for bins_v in [16,32,64,256]:
                            buscador = BuscadorDeImagensGPU(pasta_banco,
                                                            color_space=metodo_busca,
                                                            aplicar_quantizacao_imagem=aplicar_quantizacao_imagem,
                                                            aplicar_filtro=aplicar_filtro,
                                                            bins=bins_v)

                            for metodo_distancia in ["euclidiana", "bhattacharyya"]:
                                try:
                                    print_log(f"Imagem de busca: {imagem_consulta}")

                                    # Executa Busca e Avaliação
                                    resultados = buscador.buscar(query_path=imagem_consulta, N=3, metrica=metodo_distancia)
                                    taxa = buscador.avaliar_taxa_acerto(metrica=metodo_distancia, N=3)


                                    nova_linha = {
                                        "Classe_Query": f"c{c_idx:03d}",
                                        "Imagem_Referencia": f"c{c_idx:03d}_001.png",
                                        "Espaco_Cor": metodo_busca.upper(),
                                        "bins": bins_v,
                                        "Filtro_Gauss": aplicar_filtro,
                                        "Quantizacao": aplicar_quantizacao_imagem,
                                        "Metrica_Distancia": metodo_distancia.upper(),
                                        "Taxa_Acerto": taxa * 100
                                    }
                                    dados_tabela.append(nova_linha)

                                    print_log(f"\n>>> Testando classe: {c_idx:02d} | Espaço: {metodo_busca.upper()} | "
                                              f"Filtro: {aplicar_filtro} | Quant: {aplicar_quantizacao_imagem} | "
                                              f"Métrica: {metodo_distancia.upper()} | Taxa: {taxa * 100:.2f}%")

                                except Exception as e:
                                    print_log(f"Ocorreu um erro na classe {c_idx}: {e}")

        print_log(f"\n{'='*60}\nFIM DA EXECUÇÃO DA CLASSE {classe_n}\n{'='*60}\n")

df_resultados = pd.DataFrame(dados_tabela)

df_resultados.to_csv(nome_arquivo_csv, index=False, sep=';', encoding='utf-8-sig')

print("\n" + "#"*30)
print("TABELA FINAL DE RESULTADOS")
print("#"*30)
print(df_resultados)


INÍCIO DOS TESTES CBIR - DATA E HORA: 21/03/2026 03:57:59
[*] Inicializando Buscador no dispositivo: cuda
[#] Extraindo características de 864 imagens. Aguarde...
[V] Dataset preparado e carregado na GPU. Tamanho da matriz: torch.Size([864, 4096])
Imagem de busca: Vistex/c001_001.png
[*] Avaliando 54 protótipos via GPU...
[V] Avaliação concluída. Taxa de acerto: 96.30%

>>> Testando classe: 01 | Espaço: RGB | Filtro: False | Quant: False | Métrica: EUCLIDIANA | Taxa: 96.30%
Imagem de busca: Vistex/c001_001.png
[*] Avaliando 54 protótipos via GPU...
[V] Avaliação concluída. Taxa de acerto: 98.15%

>>> Testando classe: 01 | Espaço: RGB | Filtro: False | Quant: False | Métrica: BHATTACHARYYA | Taxa: 98.15%
[*] Inicializando Buscador no dispositivo: cuda
[#] Extraindo características de 864 imagens. Aguarde...
[V] Dataset preparado e carregado na GPU. Tamanho da matriz: torch.Size([864, 32768])
Imagem de busca: Vistex/c001_001.png
[*] Avaliando 54 protótipos via GPU...
[V] Avaliação concl

In [ ]:
colunas_config = ["Espaco_Cor", "Filtro_Gauss","bins", "Quantizacao", "Metrica_Distancia", "Taxa_Acerto"]
top_10_unicos = df_resultados.drop_duplicates(subset=colunas_config)
top_10_unicos = top_10_unicos.sort_values(by='Taxa_Acerto', ascending=False).head(10)

print(top_10_unicos[colunas_config])